# 01 - Data Audit CabAI

Notebook ini dipakai untuk mengecek kesiapan dataset CabAI: sumber data, label final, distribusi kelas, split, dan contoh gambar. Output dari notebook ini masuk ke bagian **Data Understanding** dan **Data Preparation** untuk demo progress.

## Scope Label

Label final proyek dibekukan menjadi 5 kelas: `healthy`, `leaf curl`, `leaf spot`, `whitefly`, dan `yellowish`. Kelas `powdery mildew` dari Roboflow dikeluarkan dari eksperimen utama karena tidak punya pasangan langsung pada benchmark Kaggle.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cabai.config import CLASS_NAMES, RAW_DATA_DIR, INTERIM_DATA_DIR, FIGURES_DIR
from cabai.data import build_manifest, summarize_manifest, write_manifest_csv

print('Project root:', PROJECT_ROOT)
print('Raw data dir:', RAW_DATA_DIR)
print('Final classes:', CLASS_NAMES)

## Lokasi Dataset

Letakkan dataset di folder berikut:

- `data/raw/roboflow_chili/` untuk dataset Roboflow training utama.
- `data/raw/kaggle_penyakit_cabai/` untuk dataset Kaggle external benchmark.

Jika Roboflow belum tersedia, Kaggle boleh dipakai dulu sebagai fallback demo progress. Catatan keterbatasannya harus dijelaskan saat presentasi.

In [ ]:
DATASET_ROOTS = {
    'roboflow_chili': RAW_DATA_DIR / 'roboflow_chili',
    'kaggle_penyakit_cabai': RAW_DATA_DIR / 'kaggle_penyakit_cabai',
}

all_rows = []
for dataset_name, root in DATASET_ROOTS.items():
    rows = build_manifest(root, dataset_name=dataset_name)
    print(f'{dataset_name}: {len(rows)} image(s) found at {root}')
    all_rows.extend(rows)

if not all_rows:
    print('\nBelum ada gambar dataset lokal.')
    print('Download Roboflow/Kaggle lalu letakkan sesuai struktur folder di atas.')
else:
    manifest_path = INTERIM_DATA_DIR / 'manifest_combined.csv'
    write_manifest_csv(all_rows, manifest_path)
    print('\nManifest saved to:', manifest_path)

In [ ]:
import pandas as pd

if all_rows:
    df = pd.DataFrame(all_rows)
    display(df.head())
    display(pd.crosstab(df['split'].replace('', 'unsplit'), df['label']))
else:
    df = pd.DataFrame(columns=['dataset', 'split', 'label', 'path'])
    display(df)

In [ ]:
summary = summarize_manifest(all_rows)
print('By label:', summary['by_label'])
print('By split:', summary['by_split'])

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def show_sample_grid(df, max_per_class=3):
    if df.empty:
        print('Tidak ada data untuk divisualisasikan.')
        return
    sample_rows = []
    for label in CLASS_NAMES:
        sample_rows.extend(df[df['label'] == label].head(max_per_class).to_dict('records'))
    n_cols = max_per_class
    n_rows = len(CLASS_NAMES)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    for ax in axes.ravel():
        ax.axis('off')
    for idx, row in enumerate(sample_rows):
        r = CLASS_NAMES.index(row['label'])
        c = idx % max_per_class
        ax = axes[r, c]
        img = Image.open(row['path']).convert('RGB')
        ax.imshow(img)
        ax.set_title(row['label'])
        ax.axis('off')
    fig.tight_layout()
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURES_DIR / 'sample_grid.png', dpi=160, bbox_inches='tight')
    return fig

show_sample_grid(df)

## Catatan Demo

- Dataset utama yang diinginkan: Roboflow untuk training.
- Dataset Kaggle disimpan sebagai external benchmark.
- Jika data belum lengkap saat demo progress, sampaikan bahwa pipeline audit sudah siap dan training awal memakai dataset yang tersedia terlebih dahulu.
- Semua klaim harus dibingkai sebagai identifikasi awal/prototype, bukan diagnosis pasti.